# Final C03 Train+Val Submission (A100)

Thin Colab orchestration notebook for the final single-model `c03_balanced_answer` run. It trains the f04 candidate-yes/no formulation with answer-index-balanced sampling on `train + val`, predicts the unlabeled test split, and writes `submission.csv`.

## 1. Mount Google Drive

Mount Google Drive before running the repo bootstrap flow.


In [4]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configure Paths

Set the repo checkout path, Drive output directory, final experiment ID, and optional data override.

Use `DATA_DIR_OVERRIDE = ""` for the standard downloaded Kaggle layout under `data/`. If you manually placed `train.csv`, `val.csv`, `test.csv`, and `images/` at the repo root, set `DATA_DIR_OVERRIDE = "."`.

In [5]:
from pathlib import Path

REPO_URL = "https://github.com/Demetri65/dl-kaggle-competition-final.git"
REPO_REF = "main"
REPO_DIR = Path("/content/dl-kaggle-competition-final")
SOURCE_ENV = Path("/content/.env")
UPLOADER_KEY = "_env_uploader"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/p2p_runs/final_c03_trainval_submission_a100"
DATA_DIR_OVERRIDE = ""
EXPERIMENT_ID = "c03_balanced_answer"
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 1
EVAL_BATCH_SIZE = 16
INFERENCE_COMPLETION_BATCH_SIZE = 16
NUM_WORKERS = 4
LOGGING_STEPS = 5


## 3. Prepare The Repo

Upload `.env` if needed, sync the repo, and run `scripts/bootstrap_colab.sh`.


In [6]:
# -- 0. Colab setup ------------------------------------------------
import subprocess
import ipywidgets as widgets
from IPython.display import display


def get_uploaded_file(uploader):
    value = uploader.value

    if isinstance(value, dict):
        filename, uploaded_file = next(iter(value.items()))
        if isinstance(uploaded_file, dict):
            content = uploaded_file.get("content", uploaded_file.get("data"))
        else:
            content = uploaded_file
    else:
        uploaded_file = value[0]
        if isinstance(uploaded_file, dict):
            filename = uploaded_file["name"]
            content = uploaded_file["content"]
        else:
            filename = uploaded_file.name
            content = uploaded_file.content

    payload = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
    return filename, payload


ready_to_bootstrap = SOURCE_ENV.exists()

if ready_to_bootstrap:
    print(f"Using existing {SOURCE_ENV}")
else:
    uploader = globals().get(UPLOADER_KEY)
    if uploader is None:
        uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
        globals()[UPLOADER_KEY] = uploader

    if not uploader.value:
        display(uploader)
        print("Select your local .env file in the upload widget above, then rerun this cell.")
    else:
        filename, payload = get_uploaded_file(uploader)
        SOURCE_ENV.write_bytes(payload)
        SOURCE_ENV.chmod(0o600)
        print(f"Saved {filename} to {SOURCE_ENV}")
        uploader.close()
        globals().pop(UPLOADER_KEY, None)
        ready_to_bootstrap = True

if ready_to_bootstrap:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    print(f"Repo synced to latest origin/{REPO_REF} at {REPO_DIR}")
    result = subprocess.run(
        ["bash", "scripts/bootstrap_colab.sh"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, end="")
        raise RuntimeError(f"scripts/bootstrap_colab.sh failed with exit code {result.returncode}")


Using existing /content/.env
Repo synced to latest origin/main at /content/dl-kaggle-competition-final
Using Kaggle CLI Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.1), please consider upgrading to the latest version (2.0.2)
Kaggle CLI 2.0.1
pixels-to-predictions.zip: Skipping, found more recently modified local copy (use --force to force download)
Extracting pixels-to-predictions.zip
Competition files downloaded to: /content/dl-kaggle-competition-final/data


## 4. Helpers

Helpers route the final run through the repo script, stream logs live, and verify the resolved config/artifact paths.

In [7]:
import json
import os
import shlex
import subprocess
from pathlib import Path

import yaml

REPO_ROOT = REPO_DIR.resolve()
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

COMMON_OVERRIDES = []
if DATA_DIR_OVERRIDE:
    COMMON_OVERRIDES.append(f"data.data_dir={DATA_DIR_OVERRIDE}")

EXPECTED_FINAL_CONFIG = {
    "experiment_id": "c03_balanced_answer",
    "parent_experiment_id": "f04_candidate_yes_no",
    "formulation.mode": "candidate_yes_no",
    "prompting.template": "candidate_yes_no",
    "sampling.mode": "balanced_answer_index",
    "lora.rank": 16,
    "lora.alpha": 32,
    "runtime.final_retrain": True,
    "runtime.predict_test": True,
    "fields": ["image", "question"],
}


def with_common_overrides(overrides):
    return [*COMMON_OVERRIDES, *overrides]


def extend_with_overrides(args, overrides):
    for override in overrides:
        args.extend(["--set", override])


def run_repo_command(args):
    command = ["python3", *args]
    env = os.environ.copy()
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    print("$", " ".join(shlex.quote(part) for part in command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    try:
        assert process.stdout is not None
        for line in process.stdout:
            output_lines.append(line)
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    output = "".join(output_lines)
    if returncode != 0:
        raise RuntimeError(f"Command failed with exit code {returncode}")
    return subprocess.CompletedProcess(command, returncode, stdout=output, stderr="")


def find_run_dir(run_root: Path, experiment_id: str) -> Path:
    candidates = sorted(run_root.glob(f"{experiment_id}_seed*"))
    candidates = [path for path in candidates if (path / "resolved_config.yaml").exists()]
    if not candidates:
        raise FileNotFoundError(f"No resolved_config.yaml found under {run_root}")
    if len(candidates) > 1:
        print(f"Multiple run dirs found; using {candidates[-1]}")
    return candidates[-1]


def _selected_fields(config):
    return [name for name, enabled in config["fields"].items() if enabled]


def _actual_config_values(config):
    return {
        "experiment_id": config["experiment_id"],
        "parent_experiment_id": config.get("parent_experiment_id"),
        "formulation.mode": config["formulation"]["mode"],
        "prompting.template": config["prompting"]["template"],
        "sampling.mode": config["sampling"]["mode"],
        "lora.rank": config["lora"]["rank"],
        "lora.alpha": config["lora"]["alpha"],
        "runtime.final_retrain": config["runtime"]["final_retrain"],
        "runtime.predict_test": config["runtime"]["predict_test"],
        "data.data_dir": config["data"]["data_dir"],
        "training.epochs": config["training"]["epochs"],
        "training.bf16": config["training"]["bf16"],
        "training.fp16": config["training"]["fp16"],
        "training.batch_size": config["training"]["batch_size"],
        "training.gradient_accumulation": config["training"]["gradient_accumulation"],
        "training.eval_batch_size": config["training"]["eval_batch_size"],
        "scoring.max_completion_batch_size": config["scoring"]["max_completion_batch_size"],
        "runtime.num_workers": config["runtime"]["num_workers"],
        "runtime.logging_steps": config["runtime"]["logging_steps"],
        "fields": _selected_fields(config),
    }


def assert_expected_final_config(config):
    actual = _actual_config_values(config)
    mismatches = []
    for key, expected in EXPECTED_FINAL_CONFIG.items():
        if actual[key] != expected:
            mismatches.append(f"{key}: expected {expected!r}, got {actual[key]!r}")
    if mismatches:
        raise AssertionError("Resolved config does not match final c03 train+val setup:\n" + "\n".join(mismatches))
    return actual


def print_config_values(values):
    for key, value in values.items():
        print(f"{key}: {value}")


def verify_resolved_config(run_root: Path):
    run_dir = find_run_dir(run_root, EXPERIMENT_ID)
    path = run_dir / "resolved_config.yaml"
    config = yaml.safe_load(path.read_text())
    values = assert_expected_final_config(config)
    print(f"Resolved config: {path}")
    print_config_values(values)
    return run_dir, config


def print_run_artifacts(run_dir: Path):
    summary_path = run_dir / "run_summary.yaml"
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing run summary: {summary_path}")
    summary = yaml.safe_load(summary_path.read_text())
    for key in (
        "run_summary_path",
        "resolved_config_path",
        "test_predictions_path",
        "submission_path",
    ):
        print(f"{key}: {summary.get(key)}")
    submission_path = summary.get("submission_path")
    if not submission_path or not Path(submission_path).exists():
        raise FileNotFoundError(f"Submission was not written: {submission_path}")
    return summary


print(f"Repo root: {REPO_ROOT}")
print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")
print(f"Experiment: {EXPERIMENT_ID}")
print(f"Final retrain: train + val, then predict test")
if DATA_DIR_OVERRIDE:
    print(f"Data override: {DATA_DIR_OVERRIDE}")


Repo root: /content/dl-kaggle-competition-final
Drive output dir: /content/drive/MyDrive/p2p_runs/final_c03_trainval_submission_a100
Experiment: c03_balanced_answer
Final retrain: train + val, then predict test


## 5. Preflight Config Check

Resolve the final command configuration before training. Stop here if any expected value is wrong.

In [8]:
preflight_overrides = with_common_overrides([
    "training.epochs=1",
    "training.bf16=true",
    "training.fp16=false",
    f"training.batch_size={TRAIN_BATCH_SIZE}",
    f"training.gradient_accumulation={GRADIENT_ACCUMULATION}",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    f"runtime.num_workers={NUM_WORKERS}",
    f"runtime.logging_steps={LOGGING_STEPS}",
    "runtime.final_retrain=true",
    "runtime.predict_test=true",
])

from src.config import load_experiment_config, selected_fields

preflight_config = load_experiment_config(
    REPO_ROOT,
    EXPERIMENT_ID,
    cli_overrides=preflight_overrides,
    output_dir=str(Path(DRIVE_OUTPUT_DIR) / "final_train_val_test"),
)
preflight_values = {
    "experiment_id": preflight_config.experiment_id,
    "parent_experiment_id": preflight_config.parent_experiment_id,
    "formulation.mode": preflight_config.formulation.mode,
    "prompting.template": preflight_config.prompting.template,
    "sampling.mode": preflight_config.sampling.mode,
    "lora.rank": preflight_config.lora.rank,
    "lora.alpha": preflight_config.lora.alpha,
    "runtime.final_retrain": preflight_config.runtime.final_retrain,
    "runtime.predict_test": preflight_config.runtime.predict_test,
    "runtime.num_workers": preflight_config.runtime.num_workers,
    "data.data_dir": preflight_config.data.data_dir,
    "fields": selected_fields(preflight_config),
}
for key, expected in EXPECTED_FINAL_CONFIG.items():
    actual = preflight_values[key]
    if actual != expected:
        raise AssertionError(f"{key}: expected {expected!r}, got {actual!r}")
print_config_values(preflight_values)


experiment_id: c03_balanced_answer
parent_experiment_id: f04_candidate_yes_no
formulation.mode: candidate_yes_no
prompting.template: candidate_yes_no
sampling.mode: balanced_answer_index
lora.rank: 16
lora.alpha: 32
runtime.final_retrain: True
runtime.predict_test: True
runtime.num_workers: 4
data.data_dir: data
fields: ['image', 'question']


## 6. Final Train+Val Test Prediction

Train `c03_balanced_answer` on `train + val`, skip validation metrics, predict the unlabeled test split, and write `submission.csv`.

In [ ]:
final_root = Path(DRIVE_OUTPUT_DIR) / "final_train_val_test"
final_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(final_root),
    "--final-retrain",
    "--predict-test",
]
extend_with_overrides(final_args, with_common_overrides([
    "training.epochs=1",
    "training.bf16=true",
    "training.fp16=false",
    f"training.batch_size={TRAIN_BATCH_SIZE}",
    f"training.gradient_accumulation={GRADIENT_ACCUMULATION}",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    f"runtime.num_workers={NUM_WORKERS}",
    f"runtime.logging_steps={LOGGING_STEPS}",
]))
run_repo_command(final_args)
run_dir, resolved_config = verify_resolved_config(final_root)
summary = print_run_artifacts(run_dir)

print("Expected final artifact:")
print(Path(summary["submission_path"]))


## 7. Recover Submission From Saved Adapter

Run this cell only if the final training cell saved `model/adapter_model.safetensors` but failed before writing `submission.csv`. It recreates `resolved_config.yaml` if needed, then runs test-only prediction from the saved adapter with conservative scoring batches.

In [9]:
failed_run_root = Path(DRIVE_OUTPUT_DIR) / "final_train_val_test"
failed_run_dir = failed_run_root / f"{EXPERIMENT_ID}_seed42"
recover_root = Path(DRIVE_OUTPUT_DIR) / "recover_test_eval"

adapter_path = failed_run_dir / "model" / "adapter_model.safetensors"
adapter_config_path = failed_run_dir / "model" / "adapter_config.json"
if not adapter_path.exists():
    raise FileNotFoundError(f"Missing saved adapter weights: {adapter_path}")
if not adapter_config_path.exists():
    raise FileNotFoundError(f"Missing saved adapter config: {adapter_config_path}")

resolved_config_path = failed_run_dir / "resolved_config.yaml"
if resolved_config_path.exists():
    print(f"Using existing resolved config: {resolved_config_path}")
else:
    from src.config import load_experiment_config

    failed_run_dir.mkdir(parents=True, exist_ok=True)
    source_config = load_experiment_config(
        REPO_ROOT,
        EXPERIMENT_ID,
        cli_overrides=with_common_overrides([
            "training.epochs=1",
            "training.bf16=true",
            "training.fp16=false",
            f"training.batch_size={TRAIN_BATCH_SIZE}",
            f"training.gradient_accumulation={GRADIENT_ACCUMULATION}",
            f"training.eval_batch_size={EVAL_BATCH_SIZE}",
            f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
            f"runtime.num_workers={NUM_WORKERS}",
            f"runtime.logging_steps={LOGGING_STEPS}",
            "runtime.final_retrain=true",
            "runtime.predict_test=true",
        ]),
        output_dir=str(failed_run_root),
    )
    with resolved_config_path.open("w") as handle:
        yaml.safe_dump(source_config.to_dict(), handle, sort_keys=False)
    print(f"Wrote missing resolved config: {resolved_config_path}")

recovery_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(recover_root),
    "--predict-test",
]
extend_with_overrides(recovery_args, with_common_overrides([
    f"runtime.eval_artifact_dir={failed_run_dir}",
    "training.epochs=0",
    "training.bf16=true",
    "training.fp16=false",
    "training.eval_batch_size=16",
    "scoring.max_completion_batch_size=16",
    "runtime.max_val_examples=0",
    f"runtime.num_workers={NUM_WORKERS}",
    f"runtime.logging_steps={LOGGING_STEPS}",
]))
run_repo_command(recovery_args)

recover_run_dir = find_run_dir(recover_root, EXPERIMENT_ID)
recover_summary = print_run_artifacts(recover_run_dir)
print("Recovered submission artifact:")
print(Path(recover_summary["submission_path"]))


Using existing resolved config: /content/drive/MyDrive/p2p_runs/final_c03_trainval_submission_a100/final_train_val_test/c03_balanced_answer_seed42/resolved_config.yaml
$ python3 scripts/run_experiment.py --experiment c03_balanced_answer --output-dir /content/drive/MyDrive/p2p_runs/final_c03_trainval_submission_a100/recover_test_eval --predict-test --set runtime.eval_artifact_dir=/content/drive/MyDrive/p2p_runs/final_c03_trainval_submission_a100/final_train_val_test/c03_balanced_answer_seed42 --set training.epochs=0 --set training.bf16=true --set training.fp16=false --set training.eval_batch_size=16 --set scoring.max_completion_batch_size=16 --set runtime.max_val_examples=0 --set runtime.num_workers=4 --set runtime.logging_steps=5
2026-04-27 18:43:25.156027: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment va

## 8. Summarize Results

Print the latest tracked runs. Final train+val runs have no validation accuracy because validation is included in training.

In [10]:
summary_args = [
    "scripts/summarize_results.py",
    "--sort-by", "timestamp",
    "--top", "20",
]
run_repo_command(summary_args)


$ python3 scripts/summarize_results.py --sort-by timestamp --top 20
                       timestamp       experiment_id parent_experiment_id  seed  run_mode  trainable_parameter_count formulation_mode  prompt_template         sampling_mode  val_accuracy  two_choice_accuracy  two_choice_support  yes_no_true_false_accuracy  yes_no_true_false_support                                                                                                  eval_artifact_dir                                                                                                      output_dir
2026-04-27T19:23:27.398748+00:00 c03_balanced_answer f04_candidate_yes_no    42 eval_only                          0 candidate_yes_no candidate_yes_no balanced_answer_index           NaN                  NaN                 NaN                         NaN                        NaN /content/drive/MyDrive/p2p_runs/final_c03_trainval_submission_a100/final_train_val_test/c03_balanced_answer_seed42 /content/drive/MyDrive/p

CompletedProcess(args=['python3', 'scripts/summarize_results.py', '--sort-by', 'timestamp', '--top', '20'], returncode=0, stdout='                       timestamp       experiment_id parent_experiment_id  seed  run_mode  trainable_parameter_count formulation_mode  prompt_template         sampling_mode  val_accuracy  two_choice_accuracy  two_choice_support  yes_no_true_false_accuracy  yes_no_true_false_support                                                                                                  eval_artifact_dir                                                                                                      output_dir\n2026-04-27T19:23:27.398748+00:00 c03_balanced_answer f04_candidate_yes_no    42 eval_only                          0 candidate_yes_no candidate_yes_no balanced_answer_index           NaN                  NaN                 NaN                         NaN                        NaN /content/drive/MyDrive/p2p_runs/final_c03_trainval_submission_a100/final_tra